# B — Điều kiện đối chứng cho Feedback Generator

**Đóng ý kiến #5** (R-b Q6, Stanford): hiện *no baseline feedback was rated*, nên điểm 3,6–4,3/5
không diễn giải được.

Sinh feedback **LLMKT-only** (chỉ hội thoại + lượt học sinh; **không** mastery, **không** chẩn đoán,
**không** next-KC) cho **đúng 30 case** đã chấm, rồi xuất phiếu chấm mù **60 mục trộn lẫn** cho
cùng hai người chấm, cùng thang 1–5.

**Cần:** GPU Colab + HF token. Chấm điểm sau đó bằng `cpu/07_score_feedback_baseline.py`.

In [ ]:
!pip -q install transformers accelerate bitsandbytes openpyxl
from huggingface_hub import notebook_login; notebook_login()

In [ ]:
import json, os, re, torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from google.colab import drive; drive.mount("/content/drive")
KIT = "/content/drive/MyDrive/multi-agents-knowledge-tracing"         # <<< ban clone repo nay tren Drive
if not os.path.exists(KIT):                                          # lan dau: tu clone tu GitHub
    import subprocess; subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/manhhdv/multi-agents-knowledge-tracing.git", KIT], check=True)
MATHKT_ROOT = KIT                                               # thu muc goc repo (chua results/, raw_runs/)
OUT_DIR = f"{KIT}/colab_out"                                    # dau ra ghi thang len Drive (khong mat khi runtime reset)
os.makedirs(OUT_DIR, exist_ok=True); os.chdir(OUT_DIR)
CASES = json.load(open(f"{MATHKT_ROOT}/results/06_feedback/01_inputs/inputs_ccmf.json"))["feedback_cases"]
assert len(CASES) == 30, len(CASES)

MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto", torch_dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.bfloat16))
model.eval()

@torch.no_grad()
def gen(prompt, max_new_tokens=320):
    enc = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)  # transformers 5 tra ve BatchEncoding
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)          # greedy, giong bai
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## Prompt đối chứng

Khác biệt DUY NHẤT so với hệ thống trong bài: bỏ mastery, bỏ chẩn đoán lỗi, bỏ next-KC.
Giữ nguyên định dạng JSON và giới hạn độ dài để so sánh công bằng.

In [ ]:
BASELINE = '''You are a mathematics tutor. Write the next tutoring message to the student.

Problem: {problem}

Conversation so far:
{history}

Student's latest turn: {student_text}

Write one short tutoring reply (2-4 sentences). Do not reveal the final answer.
Return ONLY JSON: {{"feedback": "<your reply>"}}'''

def parse(raw):
    m = re.search(r"\{.*\}", raw, re.S)
    if m:
        try: return json.loads(m.group())["feedback"].strip(), True
        except Exception: pass
    return raw.strip(), False

rows = []
for i, c in enumerate(CASES):
    raw = gen(BASELINE.format(problem=c["problem"], history=c["history"], student_text=c["student_text"]))
    txt, ok = parse(raw)
    rows.append(dict(dialogue=c["dialogue"], turn_id=c["turn_id"], verdict=c["verdict"],
                     condition="baseline_llmkt_only", feedback_text=txt, valid_json=ok, raw_output=raw))
    print(i + 1, end=" ")
B = pd.DataFrame(rows)
B.to_csv("feedback_baseline_cases.csv", index=False)
print("\nvalid JSON:", B.valid_json.mean())

## Phiếu chấm mù, 60 mục trộn lẫn

Người chấm **không** thấy cột `condition`. Thứ tự cố định bằng seed để hai phiếu A/B giống nhau.

In [ ]:
orig = pd.read_csv(f"{MATHKT_ROOT}/results/06_feedback/02_generation/feedback_generator_cases.csv")
orig = orig[["dialogue", "turn_id", "verdict", "feedback_text"]].assign(condition="system_mastery_conditioned")
ALL = pd.concat([orig, B[["dialogue", "turn_id", "verdict", "feedback_text", "condition"]]], ignore_index=True)

ctx = {(str(c["dialogue"]), int(c["turn_id"])): c for c in CASES}
ALL["de_bai"] = [ctx[(str(d), int(t))]["problem"] for d, t in zip(ALL.dialogue, ALL.turn_id)]
ALL["ngu_canh"] = [ctx[(str(d), int(t))]["history"] for d, t in zip(ALL.dialogue, ALL.turn_id)]
ALL["luot_hoc_sinh"] = [ctx[(str(d), int(t))]["student_text"] for d, t in zip(ALL.dialogue, ALL.turn_id)]
ALL["dap_an"] = [ctx[(str(d), int(t))]["reference"] for d, t in zip(ALL.dialogue, ALL.turn_id)]

ALL = ALL.sample(frac=1, random_state=20260914).reset_index(drop=True)
ALL.insert(0, "stt", range(1, len(ALL) + 1))
ALL.insert(1, "item_id", ["FB%03d" % i for i in ALL.stt])
ALL.to_csv("feedback_baseline_KEY.csv", index=False)      # GIU KIN cho den khi cham xong

SHEET = ["stt", "item_id", "de_bai", "ngu_canh", "luot_hoc_sinh", "dap_an", "feedback_text"]
for who in ("A", "B"):
    s = ALL[SHEET].copy()
    for c in ("correctness", "relevance", "clarity"): s[c] = ""
    s["lo_dap_an (co/khong)"] = ""; s["ghi_chu"] = ""
    with pd.ExcelWriter(f"feedback_baseline_rater_{who}.xlsx", engine="openpyxl") as w:
        s.to_excel(w, sheet_name="Cham diem", index=False)
        pd.DataFrame({"HUONG DAN": [
            "Cham 60 phan hoi cua he gia su, thang 1-5 cho Correctness / Relevance / Clarity.",
            "Dung CUNG tieu chi va cung thang nhu vong cham 30 case truoc do.",
            "Cac muc da duoc tron ngau nhien. KHONG co thong tin ve nguon sinh ra phan hoi.",
            "lo_dap_an: ghi 'co' neu phan hoi tiet lo dap an cuoi cung, nguoc lai ghi 'khong'.",
            "Khong dung cong cu AI. Khong trao doi giua hai nguoi cham cho den khi xong."]}
        ).to_excel(w, sheet_name="Huong dan", index=False)
print("-> feedback_baseline_cases.csv, feedback_baseline_KEY.csv (giu kin),",
      "feedback_baseline_rater_A.xlsx, feedback_baseline_rater_B.xlsx")